In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load data
df_raw = pd.read_csv('airline_route_profitability.csv')
print(f'Dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

Dataset shape: 7,974 rows x 33 columns


In [2]:
# Data quality checks
print('=== Column Data Types ===')
print(df_raw.dtypes)

=== Column Data Types ===
Flight_Number               object
Flight_Date                 object
Origin                      object
Destination                 object
Route                       object
Aircraft_Type               object
Aircraft_Capacity            int64
Passengers                   int64
Load_Factor                float64
Flight_Hours               float64
Season                      object
Route_Category              object
Demand_Level                object
Ticket_Revenue             float64
Ancillary_Revenue          float64
Total_Revenue              float64
Fuel_Cost                  float64
Maintenance_Cost           float64
Crew_Cost                  float64
Depreciation_Cost          float64
Insurance_Cost             float64
Airport_Fees               float64
Catering_Cost              float64
Handling_Cost              float64
Navigation_Fees            float64
Sales_Distribution_Cost    float64
Passenger_Service_Cost     float64
Overhead_Cost              fl

In [3]:
# Check for missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing Pct': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

                   Missing Count  Missing Pct
Ancillary_Revenue            271         3.40
Catering_Cost                265         3.32
Handling_Cost                256         3.21


In [4]:
# Check for duplicate records
n_dupes = df_raw.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')

Duplicate rows: 0


### Check Consistencies

In [5]:
# 1. Build working copy
# =========================
df = df_raw.copy()
df['Flight_Date'] = pd.to_datetime(df['Flight_Date'], errors='coerce')

# =========================
# 2. Helper settings
# =========================
EPS = 1e-6
MONEY_TOL = 0.05     # tolerance for currency-like fields
MARGIN_TOL = 0.01   # tolerance for margin percentage
LOAD_TOL = 0.01

cost_cols = [
    'Fuel_Cost', 'Maintenance_Cost', 'Crew_Cost', 'Depreciation_Cost',
    'Insurance_Cost', 'Airport_Fees', 'Catering_Cost', 'Handling_Cost',
    'Navigation_Fees', 'Sales_Distribution_Cost', 'Passenger_Service_Cost',
    'Overhead_Cost', 'Marketing_Cost', 'IT_Systems_Cost'
]

# Derived columns
df['Load_Factor_Computed'] = df['Passengers'] / df['Aircraft_Capacity']
df['Total_Revenue_Computed'] = df['Ticket_Revenue'] + df['Ancillary_Revenue']
df['Total_Cost_Computed'] = df[cost_cols].sum(axis=1, min_count=len(cost_cols))
df['Profit_Computed'] = df['Total_Revenue'] - df['Total_Cost']
df['Profit_Margin_Computed'] = np.where(
    df['Total_Revenue'].notna() & (df['Total_Revenue'].abs() > EPS),
    df['Profit_Computed'] / df['Total_Revenue'] * 100,
    np.nan
)

def compare_cols(df, col1, col2, tol=1e-6):
    mask = df[col1].notna() & df[col2].notna()
    diff = (df.loc[mask, col1] - df.loc[mask, col2]).abs()
    return {
        'rows_checked': mask.sum(),
        'max_abs_diff': diff.max() if len(diff) > 0 else np.nan,
        'rows_failed': (diff > tol).sum() if len(diff) > 0 else np.nan
    }

print("Load factor:", compare_cols(df, 'Load_Factor', 'Load_Factor_Computed', LOAD_TOL))
print("Total revenue:", compare_cols(df, 'Total_Revenue', 'Total_Revenue_Computed', MONEY_TOL))
print("Total cost:", compare_cols(df, 'Total_Cost', 'Total_Cost_Computed', MONEY_TOL))
print("Profit:", compare_cols(df, 'Profit', 'Profit_Computed', MONEY_TOL))
print("Profit margin:", compare_cols(df, 'Profit_Margin', 'Profit_Margin_Computed', MARGIN_TOL))

Load factor: {'rows_checked': np.int64(7974), 'max_abs_diff': 0.0055888888888888655, 'rows_failed': np.int64(0)}
Total revenue: {'rows_checked': np.int64(7703), 'max_abs_diff': 0.01000000024214387, 'rows_failed': np.int64(0)}
Total cost: {'rows_checked': np.int64(7453), 'max_abs_diff': 0.030000000086147338, 'rows_failed': np.int64(0)}
Profit: {'rows_checked': np.int64(7974), 'max_abs_diff': 0.010000000125728548, 'rows_failed': np.int64(0)}
Profit margin: {'rows_checked': np.int64(7974), 'max_abs_diff': 0.005012173639533657, 'rows_failed': np.int64(0)}


### Missing Value Treatment

The dataset contains missing values in only three variables: `Ancillary_Revenue`, `Catering_Cost`, and `Handling_Cost`. All aggregate fields, including `Total_Revenue` and `Total_Cost`, are complete and were first verified to be internally consistent with the accounting identities defined in the metadata.

Because `Total_Revenue = Ticket_Revenue + Ancillary_Revenue`, all missing values in `Ancillary_Revenue` were deterministically reconstructed as:

`Ancillary_Revenue = Total_Revenue - Ticket_Revenue`

For cost variables, the metadata states that `Total_Cost` is the sum of all cost components. Therefore:

- if an observation had only one missing cost component among `Catering_Cost` and `Handling_Cost`, that missing value was deterministically reconstructed from `Total_Cost` and the remaining observed cost components;
- if an observation had both `Catering_Cost` and `Handling_Cost` missing simultaneously, deterministic reconstruction was not possible, and the remaining missing values were imputed using hierarchical group medians based on operationally similar flights.

Missingness indicators were preserved, while zero imputation and row deletion were avoided.

In [6]:
# Preserve original missingness indicators
for col in ['Ancillary_Revenue', 'Catering_Cost', 'Handling_Cost']:
    df[f'{col}_was_missing'] = df[col].isna().astype(int)

# Ancillary_Revenue can always be reconstructed because: Total_Revenue = Ticket_Revenue + Ancillary_Revenue
mask_anc = df['Ancillary_Revenue'].isna()
df.loc[mask_anc, 'Ancillary_Revenue'] = (
    df.loc[mask_anc, 'Total_Revenue'] - df.loc[mask_anc, 'Ticket_Revenue']
)

# Count missing values across all cost components
df['missing_cost_count'] = df[cost_cols].isna().sum(axis=1)

# Catering_Cost can be reconstructed when it is the only missing cost component
mask_cat = (
    df['Catering_Cost'].isna()
    & (df['missing_cost_count'] == 1)
)
df.loc[mask_cat, 'Catering_Cost'] = (
    df.loc[mask_cat, 'Total_Cost']
    - df.loc[mask_cat, [c for c in cost_cols if c != 'Catering_Cost']].sum(axis=1)
)

# Handling_Cost can be reconstructed when it is the only missing cost component
mask_hand = (
    df['Handling_Cost'].isna()
    & (df['missing_cost_count'] == 1)
)
df.loc[mask_hand, 'Handling_Cost'] = (
    df.loc[mask_hand, 'Total_Cost']
    - df.loc[mask_hand, [c for c in cost_cols if c != 'Handling_Cost']].sum(axis=1)
)

# -------------------------
# Impute only rows where both Catering_Cost and Handling_Cost were missing
# -------------------------

def fill_by_group_median(df, col, group_levels):
    """Fill missing values hierarchically using group medians, then global median."""
    for group_cols in group_levels:
        group_median = df.groupby(group_cols)[col].transform('median')
        df[col] = df[col].fillna(group_median)
    df[col] = df[col].fillna(df[col].median())
    return df

df = fill_by_group_median(
    df,
    'Catering_Cost',
    [['Route_Category', 'Aircraft_Type'], ['Route_Category']]
)

df = fill_by_group_median(
    df,
    'Handling_Cost',
    [['Route_Category', 'Aircraft_Type'], ['Route_Category']]
)

# -------------------------
# Clean up helper column and verify
# -------------------------

df.drop(columns=['missing_cost_count'], inplace=True)

print(df[['Ancillary_Revenue', 'Catering_Cost', 'Handling_Cost']].isna().sum())

Ancillary_Revenue    0
Catering_Cost        0
Handling_Cost        0
dtype: int64


In [7]:
# Save the cleaned data for use in the next notebook
df.to_csv('df_preprocessed.csv', index=False)